# [실습] LangChain을 이용한 데이터 분류와 전처리

LangChain Expression Language(LCEL)는 랭체인에서 체인을 구성하는 문법입니다.    


## 라이브러리 설치  

랭체인 OpenAI 모듈을 설치합니다.

In [ ]:
%pip install langchain langchain_openai dotenv arxiv -q

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv(override=True)
# (기본값: '.env', override=True를 통해 기존 환경 변수를 덮어쓰기 가능)

if os.environ.get('OPENAI_API_KEY'):
    print('OpenAI API 키 확인')


### init_chat_model() 로 모델 불러오기

랭체인에서는 아래 코드를 통해 런타임 중의 모델 수정을 지원합니다.

In [ ]:
from langchain.chat_models import init_chat_model


gpt41 = init_chat_model(
    "gpt-4.1-mini", temperature=0.3)

gpt5 = init_chat_model(
    "gpt-5.2", reasoning_effort='low')
# claude_opus = init_chat_model(
#     "claude-4.5-opus", model_provider="anthropic", temperature=0
# )

# gemini_llm = init_chat_model(
#     "gemini-3-flash-preview", model_provider="google_genai", temperature=0
# )

prompt = '모델명과 함께 자기소개를 한줄로 부탁해. 오늘은 몇월 며칠이지?'

print("GPT4.1: " + gpt41.invoke(prompt).text + "\n")
print("GPT5: " + gpt5.invoke(prompt).text + "\n")

앞에서 배운 ChatPromptTemplate와 LLM을 연결해 체인을 구성합니다.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

fun_chat_template = ChatPromptTemplate([
    ('user', """
### Role
당신은 영어와 한국어의 번역에 능통한 유머의 달인입니다.

### Instruction
1.  먼저, [{topic}]에 관한 영어 Pun 농담을 하나 제시하세요.
해당 농담은 한국어로 번역했을 때에서도 그 의미가 통하고 유머가 유지될 수 있어야 합니다.
만약 직역이 어렵다면, 창의적으로 각색하여 한국어 버전의 농담을 출력하세요.
- 한국어의 유사 발음, 단어의 중의적 의미, 혹은 한국의 문화적 상황 등을 활용할 수 있습니다.
2.  다음으로, 해당 농담이 영어 원어민 사용자에게 왜 재미있는지 그들의 언어적 유희 및 문화적 관점에서 한국어로 설명하세요.
""")])

-----------
LCEL의 구조에서는 템플릿과 llm 모델을 설정하고, 이를 하나로 묶어 체인을 생성합니다.

In [ ]:
joke = fun_chat_template | gpt5

이후, 체인의 invoke를 실행하며 입력 포맷을 전달하면, 순서대로 체인이 실행되며 최종 결과로 연결됩니다.       

입력 변수가 프롬프트 템플릿에 전달되고, 완성된 프롬프트가 LLM에 들어가는 구조입니다.  
입력 포맷은 Dict 형식으로 전달합니다.

In [ ]:
response = joke.invoke({'topic':'eggs'})
# 매개변수가 1개일 때는 joke.invoke('eggs') 도 가능
print(response.text)

In [ ]:
response = joke.invoke({'topic':'pigeon', 'foo':'bar'})
# 프롬프트에 포함되어 있지 않은 매개변수는 무시
print(response.text)

In [ ]:
# 체인이 LLM에 전달하는 실체
fun_chat_template.invoke({'topic':'eggs'}).messages

## [실습] 매개변수가 2개인 Prompt-LLM Chain 생성하기   
임의의 ChatPromptTemplate를 만들고, 2개의 매개변수를 받도록 구성하여 체인을 만들고 실행하세요.

In [ ]:
# 아래 LLM을 사용하세요!
gpt5 = init_chat_model(
    "gpt-5.2", reasoning_effort='low')

In [ ]:
prompt = None

In [ ]:
chain = None

In [ ]:
# chain.invoke()

<br><br><br><br><br><br><br><br><br><br><br><br>

In [ ]:
prompt = ChatPromptTemplate(
    [
        ('system', '당신은 재미있고 교훈적인 이야기를 씁니다.'),
        ('user', '{A}와 {B}가 만났을 때의 대화를 써 주세요.')
    ])
chain = prompt | gpt5
response = chain.invoke({'A':'햄릿', 'B':'슈퍼마리오'})
print(response.text)

### Prompt | LLM | Parser 체인

LCEL의 체인에는 파서(Parser)를 추가할 수 있습니다.    
파서는 출력 형식을 변환합니다.

StrOutputParser : 출력 결과를 String 형식으로 변환합니다.

In [ ]:
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()

recipe_template=ChatPromptTemplate([
    ('system','당신은 전세계의 조리법을 아는 쉐프입니다.'),
    ('user','''저는 다음의 재료를 이용한 환상적인 요리를 만들고 싶습니다.

레시피와 함께, 고객의 시선을 사로잡을 수 있는 추천사도 작성해 주세요.
---
[재료]: {ingredient}''')
])

In [ ]:
recipe_chain = recipe_template | gpt41 | parser
response = recipe_chain.invoke({'ingredient':'연두부, 에너지바, 바나나'})
print(response)

## [실습] 검색 결과 분류 체인 만들기

다음은 Arxiv의 최신 논문을 검색하는 함수입니다.   
해당 논문들이 LLM 관련 논문인지 분류하는 체인을 만들고, 실행하여 결과를 비교하세요.   
함수의 결과물로 다양한 값들이 있으므로, 값들 중 필요한 값만 입력받는 체인을 만들고 실행하세요.

In [ ]:
import arxiv
from typing import List, Dict, Optional

def get_arxiv_papers(query: Optional[str] = None, N: int = 10) -> List[Dict]:
    """
    arXiv에서 논문 리스트를 가져오는 함수

    Parameters:
    -----------
    query : str, optional
        검색어
    N : int, default=10
        가져올 논문 개수

    Returns:
    --------
    List[Dict] : 논문 정보를 담은 딕셔너리 리스트
    """


    search_query = query

    # arxiv 클라이언트 생성
    client = arxiv.Client()

    # 검색 객체 생성
    search = arxiv.Search(
        query=search_query,
        max_results=N,
        sort_by=arxiv.SortCriterion.SubmittedDate,  # 제출일 기준 정렬
        sort_order=arxiv.SortOrder.Descending  # 최신순
    )

    # 결과를 저장할 리스트
    papers = []

    # 검색 실행 (새로운 API 사용)
    for result in client.results(search):
        paper_info = {
            'title': result.title,
            'authors': [author.name for author in result.authors],
            'summary': result.summary,
            'published': result.published.strftime('%Y-%m-%d %H:%M:%S'),
            'updated': result.updated.strftime('%Y-%m-%d %H:%M:%S'),
            'arxiv_id': result.entry_id.split('/')[-1],  # arXiv ID 추출
            'pdf_url': result.pdf_url,
            'categories': result.categories,
            'primary_category': result.primary_category,
            'comment': result.comment,
            'journal_ref': result.journal_ref
        }
        papers.append(paper_info)

    return papers

query = 'Security'
print(f"\n\n=== 검색어 `{query}` 로 검색한 최근 논문 ===")
security_papers = get_arxiv_papers(query=query, N=5)
print(f"총 {len(security_papers)}개의 논문을 가져왔습니다.")
print('\n'.join([paper['title'] for paper in security_papers]))

# 임의의 검색어로 검색하려면 query를 바꿔 다시 호출하세요.


In [ ]:
classify_chain = None

## [실습] LLM 최신 연구 요약 체인 만들기

분류 결과를 바탕으로, LLM 관련 논문만 모아 요약할 수 있습니다.

적절한 요약 프롬프트를 생성하여, 이전 실습의 결과 중 LLM에 해당하는 결과들만을 모으세요.

In [ ]:
LLM_documents=[]

# LLM 분류 조건 만족시, LLM_documents에 정보 저장


LLM_documents

In [ ]:
# 요약 프롬프트와 체인 만들기
summary_prompt = None
summary_chain = None

In [ ]:
# LLM 페이퍼 요약 출력하기
